# 01 — TSMOM Baseline

## <b>Block 01 — Data Audit</b>

### Objetivo
Verificar que el universo diario Gold está completo, consistente y alineado antes de calcular retornos o cualquier objeto de estrategia.

### Alcance
Universo: `SPY`, `QQQ`, `XLE`, `GLD`. Ventana solicitada: `2018-01-02` hasta la última sesión común disponible no posterior a `2026-11-17`.

### Fuente y causalidad
Este notebook lee exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. No se permite look-ahead: las etapas posteriores deberán utilizar únicamente información disponible en cada instante de decisión.


In [19]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

SYMBOLS = ("SPY", "QQQ", "XLE", "GLD")
WINDOW_START = date(2018, 1, 2)
WINDOW_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
GOLD_ROOT


WindowsPath('C:/Users/avazq/OneDrive/Escritorio/cienciadatos/portafolio/quant-research-notes/Momentum_systematic_strategies/data/gold/daily_series/etf')

In [20]:
REQUIRED_COLUMNS = (
    "session_date", "open", "high", "low", "close", "adjusted_close",
    "volume", "dividend_amount", "split_coefficient",
)

frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    missing_columns = set(REQUIRED_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{symbol} no contiene columnas requeridas: {sorted(missing_columns)}")
    frames[symbol] = frame.filter(
        pl.col("session_date").is_between(WINDOW_START, WINDOW_END, closed="both")
    ).sort("session_date")

{symbol: frame.shape for symbol, frame in frames.items()}


{'SPY': (2232, 14), 'QQQ': (2232, 14), 'XLE': (2232, 14), 'GLD': (2232, 14)}

In [21]:
def ohlc_violation_count(frame: pl.DataFrame) -> int:
    invalid_bar = (
        (pl.col("open") <= 0) | (pl.col("high") <= 0) |
        (pl.col("low") <= 0) | (pl.col("close") <= 0) |
        (pl.col("high") < pl.col("low")) |
        (pl.col("high") < pl.col("open")) |
        (pl.col("high") < pl.col("close")) |
        (pl.col("low") > pl.col("open")) |
        (pl.col("low") > pl.col("close"))
    )
    return frame.select(invalid_bar.sum()).item()

audit_rows = []
for symbol, frame in frames.items():
    null_counts = frame.select(pl.all().null_count()).to_dicts()[0]
    audit_rows.append({
        "symbol": symbol,
        "sessions": frame.height,
        "start": frame.get_column("session_date").min(),
        "end": frame.get_column("session_date").max(),
        "nulls_total": sum(null_counts.values()),
        "duplicate_dates": frame.select(pl.col("session_date").is_duplicated().sum()).item(),
        "ohlc_violations": ohlc_violation_count(frame),
        "adjusted_close_available": "adjusted_close" in frame.columns,
        "positive_dividend_sessions": frame.select((pl.col("dividend_amount") > 0).sum()).item(),
        "non_unit_split_sessions": frame.select((pl.col("split_coefficient") != 1).sum()).item(),
    })

asset_audit = pl.DataFrame(audit_rows)
asset_audit


symbol,sessions,start,end,nulls_total,duplicate_dates,ohlc_violations,adjusted_close_available,positive_dividend_sessions,non_unit_split_sessions
str,i64,date,date,i64,i64,i64,bool,i64,i64
"""SPY""",2232,2018-01-02,2026-11-17,0,0,0,true,35,0
"""QQQ""",2232,2018-01-02,2026-11-17,0,0,0,true,36,0
"""XLE""",2232,2018-01-02,2026-11-17,0,0,0,true,36,1
"""GLD""",2232,2018-01-02,2026-11-17,0,0,0,true,0,0


In [22]:
panel = None
for symbol in SYMBOLS:
    series = frames[symbol].select(
        "session_date", pl.col("adjusted_close").alias(symbol)
    )
    panel = series if panel is None else panel.join(series, on="session_date", how="inner")

panel = panel.sort("session_date")
panel_nulls = panel.select(pl.all().null_count()).to_dicts()[0]
panel_summary = {
    "rows": panel.height,
    "columns": panel.width,
    "start": panel.get_column("session_date").min(),
    "end": panel.get_column("session_date").max(),
    "nulls_total": sum(panel_nulls.values()),
    "duplicate_dates": panel.select(pl.col("session_date").is_duplicated().sum()).item(),
}

assert panel.columns == ["session_date", *SYMBOLS]
assert panel_summary["start"] == WINDOW_START
assert panel_summary["end"] <= WINDOW_END
assert panel_summary["nulls_total"] == 0

print(panel_summary)
panel.head()


{'rows': 2232, 'columns': 5, 'start': datetime.date(2018, 1, 2), 'end': datetime.date(2026, 11, 17), 'nulls_total': 0, 'duplicate_dates': 0}


session_date,SPY,QQQ,XLE,GLD
date,f64,f64,f64,f64
2018-01-02,235.20385,149.695763,25.380449,125.15
2018-01-03,236.69154,151.150312,25.760551,124.82
2018-01-04,237.689168,151.414775,25.916047,125.46
2018-01-05,239.273121,152.93544,25.905681,125.33
2018-01-08,239.710677,153.530483,26.061177,125.31


In [23]:
normalized_panel = panel.with_columns([
    (pl.col(symbol) / pl.col(symbol).first()).alias(symbol)
    for symbol in SYMBOLS
])

figure = go.Figure()
for symbol in SYMBOLS:
    figure.add_scatter(
        x=normalized_panel.get_column("session_date").to_list(),
        y=normalized_panel.get_column(symbol).to_list(),
        mode="lines",
        name=symbol,
    )

figure.update_layout(
    title="Normalized adjusted close (P_t / P_0)",
    xaxis_title="Session date",
    yaxis_title="P_t / P_0",
    template="plotly_white",
    legend_title="ETF",
)
figure.show()


### Resultados de la ejecución — 2026-11-17

| Activo | Sesiones | Rango | Nulos | Fechas duplicadas | Violaciones OHLC | `adjusted_close` | Dividendos positivos | Splits no unitarios |
| --- | ---: | --- | ---: | ---: | ---: | --- | ---: | ---: |
| SPY | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 35 | 0 |
| QQQ | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 0 |
| XLE | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 1 |
| GLD | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 0 | 0 |

La ventana común exacta es `2018-01-02 → 2026-11-17`. El panel wide `session_date | SPY | QQQ | XLE | GLD` tiene dimensión **2,232 × 5**, sin nulos ni fechas duplicadas. El split no unitario de XLE queda registrado como dato corporativo informativo; no constituye una violación OHLC.

Los datos quedan listos para continuar con Block 02.


## <b>Block 02 — Returns</b>

> Los retornos se calculan sobre el panel Gold limpio de Block 01. La medida principal es el retorno logarítmico, no el retorno porcentual simple.

$ r_t = \ln(P_t / P_{t-1}) $

Para cada horizonte `k`, el momentum continuo usa $ r_t^{(k)} = \ln(P_t / P_{t-k}) $. Todos estos valores usan exclusivamente precios conocidos hasta la fecha `t`; este bloque no usa retornos futuros ni crea posiciones.

In [24]:
MOMENTUM_HORIZONS = (21, 63, 126, 252)

return_columns = []
for symbol in SYMBOLS:
    log_price = pl.col(symbol).log()
    return_columns.append(log_price.diff().alias(f"{symbol}_ret_1d"))
    for horizon in MOMENTUM_HORIZONS:
        return_columns.append(
            (log_price - log_price.shift(horizon)).alias(f"{symbol}_mom_{horizon}")
        )

returns_panel = panel.with_columns(return_columns)
returns_panel.select("session_date", *[f"{symbol}_ret_1d" for symbol in SYMBOLS]).tail(3)


session_date,SPY_ret_1d,QQQ_ret_1d,XLE_ret_1d,GLD_ret_1d
date,f64,f64,f64,f64
2026-11-13,-0.003639,-0.012105,0.009746,0.001577
2026-11-16,-0.0193,-0.027043,-0.019744,-0.007115
2026-11-17,-0.010201,-0.00728,-0.005195,-0.001276


In [25]:
daily_return_audit = pl.DataFrame([
    {
        "asset": symbol,
        "mean daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").mean() * 100).item(),
        "daily volatility (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").std() * 100).item(),
        "minimum daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").min() * 100).item(),
        "maximum daily return (%)": returns_panel.select(pl.col(f"{symbol}_ret_1d").max() * 100).item(),
        "valid observations": returns_panel.select(pl.col(f"{symbol}_ret_1d").count()).item(),
    }
    for symbol in SYMBOLS
]).with_columns(pl.all().exclude(["asset", "valid observations"]).round(4))

daily_return_audit


asset,mean daily return (%),daily volatility (%),minimum daily return (%),maximum daily return (%),valid observations
str,f64,f64,f64,f64,i64
"""SPY""",0.0509,1.1993,-11.5887,9.9863,2231
"""QQQ""",0.0663,1.5068,-12.7592,11.3356,2231
"""XLE""",0.041,1.9723,-22.491,14.8742,2231
"""GLD""",0.0464,1.087,-10.8412,6.1647,2231


In [26]:
latest_horizons = pl.DataFrame([
    {
        "asset": symbol,
        **{
            f"{horizon}d momentum (%)": returns_panel.select(
                pl.col(f"{symbol}_mom_{horizon}").tail(1) * 100
            ).item()
            for horizon in MOMENTUM_HORIZONS
        },
    }
    for symbol in SYMBOLS
]).with_columns(pl.all().exclude("asset").round(4))

valid_observations = {
    horizon: returns_panel.select(pl.col(f"SPY_mom_{horizon}").count()).item()
    for horizon in MOMENTUM_HORIZONS
}

print(f"Latest date: {returns_panel.get_column('session_date').max()}")
print(f"Valid observations per horizon: {valid_observations}")
latest_horizons


Latest date: 2026-11-17
Valid observations per horizon: {21: 2211, 63: 2169, 126: 2106, 252: 1980}


asset,21d momentum (%),63d momentum (%),126d momentum (%),252d momentum (%)
str,f64,f64,f64,f64
"""SPY""",-4.5427,-4.6311,0.3263,9.6941
"""QQQ""",-8.9363,-8.5171,-6.3541,8.0661
"""XLE""",-6.7729,0.2604,4.638,34.7979
"""GLD""",-7.4361,-16.0486,-15.4815,-6.4489


### Información antes de discretizar

Los momentum returns son variables continuas: `+0.5%` y `+25%` contienen información de magnitud muy distinta. En Block 03, una señal clásica $ sign(r_t^{(k)}) $ convertiría ambas observaciones en `+1`; esa pérdida de información se documenta aquí, pero la señal no se implementa todavía.

In [27]:
spy_momentum = returns_panel.select(
    "session_date", *[f"SPY_mom_{horizon}" for horizon in MOMENTUM_HORIZONS]
)

figure = go.Figure()
for horizon in MOMENTUM_HORIZONS:
    figure.add_scatter(
        x=spy_momentum.get_column("session_date").to_list(),
        y=spy_momentum.get_column(f"SPY_mom_{horizon}").to_list(),
        mode="lines",
        name=f"{horizon}d",
    )

figure.add_hline(y=0, line_color="black", line_width=1)
figure.update_layout(
    title=(
        "SPY multi-horizon momentum returns"
        "<br><sup>Continuous momentum measures before signal discretization</sup>"
    ),
    xaxis_title="Session date",
    yaxis_title="Log momentum return",
    template="plotly_white",
    legend_title="Lookback",
)
figure.show()


### Resultados de la ejecución — 2026-11-17

| Activo | Media diaria | Volatilidad diaria | Mínimo diario | Máximo diario | Observaciones válidas |
| --- | ---: | ---: | ---: | ---: | ---: |
| SPY | 0.0509% | 1.1993% | -11.5887% | 9.9863% | 2,231 |
| QQQ | 0.0663% | 1.5068% | -12.7592% | 11.3356% | 2,231 |
| XLE | 0.0410% | 1.9723% | -22.4909% | 14.8742% | 2,231 |
| GLD | 0.0464% | 1.0870% | -10.8412% | 6.1647% | 2,231 |

Última fecha analizada: `2026-11-17`.

| Activo | 21d | 63d | 126d | 252d |
| --- | ---: | ---: | ---: | ---: |
| SPY | -4.5427% | -4.6311% | 0.3263% | 9.6941% |
| QQQ | -8.9363% | -8.5171% | -6.3541% | 8.0661% |
| XLE | -6.7729% | 0.2604% | 4.6380% | 34.7979% |
| GLD | -7.4361% | -16.0485% | -15.4815% | -6.4489% |

### <b>Block 02 conclusion</b>

- Los retornos diarios logarítmicos se calcularon para los cuatro activos; cada uno conserva 2,231 observaciones válidas.
- Los cuatro horizontes están disponibles: 21d (2,211 observaciones), 63d (2,169), 126d (2,106) y 252d (1,980).
- La pérdida por el lookback de 252 sesiones deja 1,980 observaciones útiles por activo.
- Los horizontes discrepan en signo y magnitud: al cierre, SPY es negativo a 21d y 63d, pero positivo a 126d y 252d.
- Las variables continuas están listas para construir señales TSMOM en el siguiente bloque.

## <b>Block 03 — TSMOM Signals</b>
_Pendiente. No implementado en esta sesión._

## <b>Block 04 — Causal PnL</b>
_Pendiente. No implementado en esta sesión._

## <b>Block 05 — Diagnostics</b>
_Pendiente. No implementado en esta sesión._
